# Logs CPF Intergrall VF1 — Validation

**Migrated from:** Alteryx workflow `Logs CPF Intergrall VF1.yxmd`
**Migration date:** 2026-08-06
**Validates:** `{gold}.logs_cpf_intergrall`

## RISCO CONHECIDO — sem baseline do Alteryx

Não há arquivo de saída esperada para este indicador. Conforme a Regra 3 do
skill, isto fica **documentado como risco** e a validação passa a ser por
profiling e invariantes, **não** por comparação contra a saída real do Alteryx.

**O que isto significa:** os testes abaixo provam que o pipeline é
internamente consistente e que cada etapa faz o que o XML manda. **Não** provam
paridade numérica com o Alteryx. Duas classes de erro sobrevivem a este
notebook:

* divergência de intenção (o anti-join, ponto 1 do notebook 03);
* divergência de dados na origem (o `.xlsx` de RH mudou desde a última execução
  do Alteryx).

**Como fechar o risco:** rodar o workflow Alteryx original uma vez, guardar o
`dd-MM-yyyy.xlsx` produzido num Volume e preencher o widget `baseline_path`.
A célula de comparação abaixo ativa-se sozinha e roda os 6 checks do skill.

In [0]:
%run ./00_config

In [0]:
import pyspark.sql.functions as F

## Framework de validação (do skill, sem alterações)

In [0]:
def validate_migration(df_expected, df_migrated, key_columns=None):
    """
    Comprehensive validation of migrated output against expected Alteryx output.
    Returns a dict of validation results.
    """
    results = {}

    # 1. Row Count Comparison
    expected_count = df_expected.count()
    migrated_count = df_migrated.count()
    results["row_count"] = {
        "expected": expected_count,
        "migrated": migrated_count,
        "match": expected_count == migrated_count,
        "diff": migrated_count - expected_count,
    }

    # 2. Schema Comparison
    expected_cols = set(df_expected.columns)
    migrated_cols = set(df_migrated.columns)
    results["schema"] = {
        "missing_in_migrated": expected_cols - migrated_cols,
        "extra_in_migrated": migrated_cols - expected_cols,
        "match": expected_cols == migrated_cols,
    }

    # 3. Data Type Comparison (on common columns)
    common_cols = expected_cols & migrated_cols
    expected_types = {f.name: str(f.dataType) for f in df_expected.schema.fields if f.name in common_cols}
    migrated_types = {f.name: str(f.dataType) for f in df_migrated.schema.fields if f.name in common_cols}
    type_mismatches = {
        c: {"expected": expected_types[c], "migrated": migrated_types[c]}
        for c in common_cols
        if expected_types.get(c) != migrated_types.get(c)
    }
    results["data_types"] = {"mismatches": type_mismatches, "match": len(type_mismatches) == 0}

    # 4. Null Count Comparison
    null_comparison = {}
    for col in sorted(common_cols):
        exp_nulls = df_expected.filter(F.col(col).isNull()).count()
        mig_nulls = df_migrated.filter(F.col(col).isNull()).count()
        if exp_nulls != mig_nulls:
            null_comparison[col] = {"expected_nulls": exp_nulls, "migrated_nulls": mig_nulls}
    results["null_counts"] = {"discrepancies": null_comparison, "match": len(null_comparison) == 0}

    # 5. Numeric Aggregation Comparison
    numeric_cols = [
        f.name
        for f in df_expected.schema.fields
        if str(f.dataType)
        in ("DoubleType", "FloatType", "IntegerType", "LongType", "DecimalType(38,18)", "ShortType")
        and f.name in common_cols
    ]
    agg_comparison = {}
    for col in numeric_cols:
        exp_stats = df_expected.select(
            F.sum(col).alias("sum"), F.avg(col).alias("avg"),
            F.min(col).alias("min"), F.max(col).alias("max"),
        ).collect()[0]
        mig_stats = df_migrated.select(
            F.sum(col).alias("sum"), F.avg(col).alias("avg"),
            F.min(col).alias("min"), F.max(col).alias("max"),
        ).collect()[0]
        diffs = {}
        for stat in ["sum", "avg", "min", "max"]:
            e, m = exp_stats[stat], mig_stats[stat]
            if e is not None and m is not None:
                if abs(float(e) - float(m)) > 1e-6:
                    diffs[stat] = {"expected": float(e), "migrated": float(m)}
            elif e != m:
                diffs[stat] = {"expected": e, "migrated": m}
        if diffs:
            agg_comparison[col] = diffs
    results["numeric_aggregations"] = {"discrepancies": agg_comparison, "match": len(agg_comparison) == 0}

    # 6. Row-Level Diff (if key columns provided)
    if key_columns and all(c in common_cols for c in key_columns):
        only_in_expected = df_expected.join(df_migrated, on=key_columns, how="left_anti")
        only_in_migrated = df_migrated.join(df_expected, on=key_columns, how="left_anti")
        results["row_diff"] = {
            "rows_only_in_expected": only_in_expected.count(),
            "rows_only_in_migrated": only_in_migrated.count(),
            "match": only_in_expected.count() == 0 and only_in_migrated.count() == 0,
        }

    all_passed = all(v.get("match", True) for v in results.values())
    results["overall"] = "PASS" if all_passed else "FAIL"
    return results


def report(results):
    print(f"Overall: {results['overall']}")
    for check, detail in results.items():
        if check != "overall":
            status = "PASS" if detail.get("match", True) else "FAIL"
            print(f"  {check}: {status}")
            if not detail.get("match", True):
                for k, v in detail.items():
                    if k != "match":
                        print(f"    {k}: {v}")

## Comparação contra baseline — ativa quando `baseline_path` está preenchido

In [0]:
# Gold agora usa data real do log (data_convertida) como data_execucao.
# Valida a janela do job (start_date..end_date).
df_migrated = (
    spark.table(T_GOLD)
    .filter(
        (F.col("data_execucao") >= F.lit(str(START_DATE)))
        & (F.col("data_execucao") <= F.lit(str(END_DATE)))
    )
    .drop("data_execucao")
)

if BASELINE_PATH:
    df_expected = read_landing(BASELINE_PATH).withColumnRenamed("Chave cons.", "chave_cons")
    print(f"baseline: {BASELINE_PATH}\n")
    results = validate_migration(df_expected, df_migrated, key_columns=["log_itgl_seq"])
    report(results)
    assert results["overall"] == "PASS", "validação contra o baseline FALHOU — ver detalhes acima"
else:
    print("SEM BASELINE — comparação ignorada; profiling + invariantes abaixo.")
    print("Para ativar: preencha o widget 'baseline_path' com o .xlsx do Alteryx.")
    results = None

## Profiling (substituto por ausência de baseline, Regra 3)

Estatísticas num único `.select()` (item 9 de performance do skill).

In [0]:
prof = df_migrated.select(
    F.count("*").alias("linhas"),
    F.countDistinct("log_itgl_seq").alias("seq_distintos"),
    F.countDistinct("chave_cons").alias("chaves_distintas"),
    F.countDistinct("log_itgl_usu").alias("usuarios_distintos"),
    F.countDistinct("log_itgl_cpf_cns").alias("cpfs_consultados"),
    F.min("log_itgl_dat").alias("dat_min"),
    F.max("log_itgl_dat").alias("dat_max"),
    *[
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"nulos_{c}")
        for c in ["log_itgl_usu", "log_itgl_nm_cns", "log_itgl_cpf_cns", "log_itgl_dat", "chave_cons"]
    ],
).collect()[0].asDict()

print(f"=== profiling {T_GOLD} — execução {HOJE} ===")
for k, v in prof.items():
    print(f"  {k:24s} {v}")

## Invariantes — o que o XML garante e que deve valer sempre

In [0]:
checks = {}
n = prof["linhas"]

# 1. formato de data: o Tool 23 troca / por -
checks["data_normalizada"] = (
    df_migrated.filter(F.col("log_itgl_dat").contains("/")).count() == 0
)

# 2. (removido) janela_desde_ontem — filtro agora é configurável via start_date/end_date

# 3. Integridade: toda chave_cons presente na Gold existe no Silver
log_silver = spark.table(T_SILVER_LOG)
chaves_orfas = (
    df_migrated.select("chave_cons").distinct()
    .join(log_silver.select("chave_cons").distinct(), on="chave_cons", how="left_anti")
    .count()
)
checks["chaves_presentes_no_silver"] = chaves_orfas == 0

# 4. Inner Join: todo registro na Gold TEM match com um diretor no RH
rh = spark.table(T_SILVER_RH)
match_count = df_migrated.join(
    F.broadcast(rh),
    F.upper(df_migrated["log_itgl_nm_cns"]) == F.upper(rh["nome_funcionario"]),
    "inner"
).count()
checks["inner_join_consistente"] = match_count == df_migrated.count()

# 5. chave consistente com os componentes
checks["chave_consistente"] = (
    df_migrated.filter(
        F.col("chave_cons")
        != F.concat(
            F.coalesce("log_itgl_usu", F.lit("")),
            F.coalesce("log_itgl_nm", F.lit("")),
            F.coalesce("log_itgl_nm_cns", F.lit("")),
            F.coalesce("log_itgl_cpf_cns", F.lit("")),
            # a chave usa a data ORIGINAL com /, revertida aqui
            F.regexp_replace("log_itgl_dat", "-", "/"),
        )
    ).count()
    == 0
)

# 6. sem duplicação de partição (replaceWhere funcionou)
checks["sem_duplicacao_execucao"] = prof["seq_distintos"] == n

print("=== invariantes ===")
for k, ok in checks.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {k}")

falhas = [k for k, ok in checks.items() if not ok]
assert not falhas, f"invariantes violadas: {falhas}"

## Spot-check — 10 linhas para conferência manual

Substituto direto do baseline: a auditoria confere estas linhas contra o
Intergrall e confirma que cada uma é de facto uma consulta repetida feita por
alguém fora dos cargos de direção.

In [0]:
display(
    df_migrated.select(
        "log_itgl_seq", "log_itgl_usu", "log_itgl_nm_cns",
        "log_itgl_cpf_cns", "log_itgl_dat", "log_itgl_hor",
    ).orderBy("log_itgl_usu", "log_itgl_dat").limit(10)
)

## Distribuição por usuário — onde a auditoria olha primeiro

In [0]:
display(
    df_migrated.groupBy("log_itgl_usu")
    .agg(
        F.count("*").alias("consultas_repetidas"),
        F.countDistinct("log_itgl_cpf_cns").alias("cpfs_distintos"),
    )
    .orderBy(F.desc("consultas_repetidas"))
    .limit(20)
)

## Sumário

In [0]:
status = "PASS" if not falhas else "FAIL"
print(f"=== VALIDAÇÃO: {status} ===")
print(f"tabela        : {T_GOLD}")
print(f"execução      : {HOJE}")
print(f"linhas        : {n:,}")
print(f"invariantes   : {len(checks) - len(falhas)}/{len(checks)} PASS")
print(f"baseline      : {'comparado — ' + results['overall'] if results else 'AUSENTE (risco documentado)'}")
if not BASELINE_PATH:
    print()
    print("RISCO ABERTO: sem baseline do Alteryx, a paridade numérica com o fluxo")
    print("original não está provada. Ver a nota no topo deste notebook.")

In [0]:
cleanup_temp_tables()

dbutils.notebook.exit(f"VALIDATION {status} | {n} linhas | baseline={'sim' if BASELINE_PATH else 'nao'}")